In [1]:
from pynq.overlays.base import BaseOverlay
import time
base = BaseOverlay("base.bit")

In [2]:
%%microblaze base.PMODB
#include "gpio.h"
#include "pyprintf.h"
//Function to turn on/off a selected pin of PMODB
void write_gpio(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
}
//Function to read the value of a selected pin of PMODB
unsigned int read_gpio(unsigned int pin){
    gpio pin_in = gpio_open(pin);
    gpio_set_direction(pin_in, GPIO_IN);
    return gpio_read(pin_in);
}

In [3]:
PMOD_B_PIN_ID=7
ON=1
OFF=0
btns = base.btns_gpio
#frequency=100
def play_buzzer_until_btn_press():
    while btns.read() == 0:
        #time.sleep(0.01) #to catch button press
        write_gpio(PMOD_B_PIN_ID, ON) #
        time.sleep(1)
        write_gpio(PMOD_B_PIN_ID, OFF) #
        time.sleep(1)
print("exited")
#play_buzzer_until_btn_press()

def play_buzzer_until_timeout(duration):
    start_time = time.time()
    currentTime = start_time
    while currentTime-start_time <= duration:
        #time.sleep(0.01) #to catch button press
        write_gpio(PMOD_B_PIN_ID, ON) #
        time.sleep(duration)
        write_gpio(PMOD_B_PIN_ID, OFF) #
        time.sleep(duration/10)
        currentTime= time.time()
print("exited")
#play_buzzer_until_btn_press()

exited
exited


In [4]:
import socket

# Server information
server_ip = "127.0.0.1"  # Localhost IP address
server_port = 12345  # Port where the server is listening

def udp_socket_client():
    
    while True:
        time.sleep(0.1)
        if btns[0].read() != 0:
            # Create a UDP socket
            client_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
            print("connected to server") # replace with TCP connect code
        if btns[1].read() != 0:
            # Message to send to the server
            message = "Hello, Server!"

            # Send the message to the server
            client_socket.sendto(message.encode(), (server_ip, server_port))

            # Receive the response from the server
            response, server_address = client_socket.recvfrom(1024)  # Buffer size is 1024 bytes
            print(f"Received from server: {response.decode()}")

        if btns[2].read() != 0:
            # Close the socket
            client_socket.close()
            print("client closed")
            break

def udp_socket_server():

    # Create a UDP socket
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    server_socket.settimeout(2)  # Set timeout to 5 seconds
    # Bind the socket to the server address and port
    server_socket.bind((server_ip, server_port))

    print(f"Server listening on {server_ip}:{server_port}")

    #while btns[2].read() == 0:
    while True:
        time.sleep(0.1)
        try:
            # Wait for a message from the client
            message, client_address = server_socket.recvfrom(1024)  # Buffer size is 1024 bytes
            print(f"Received message: {message.decode()} from {client_address}")
            play_buzzer_until_timeout(0.5)
            # Send a response back to the client
            response = f"Hello from server to {client_address}".encode()
            server_socket.sendto(response, client_address)
        except socket.timeout:
            #print("Timeout occurred, no data received")
            pass
        time.sleep(0.1)
        if btns[2].read() != 0:
            #finally:
            server_socket.close()
            break
            print("server stopped")

In [5]:
import multiprocessing
import time

def task(name):
    print(f"Process {name} started")
    time.sleep(2)
    print(f"Process {name} finished")

if __name__ == "__main__":
    process1 = multiprocessing.Process(target=udp_socket_client)
    process2 = multiprocessing.Process(target=udp_socket_server)


    process2.start()
    process1.start()

    process1.join()
    print("client closed and joined")
    process2.join()
    print("server closed")

    print("Both processes finished")

Server listening on 127.0.0.1:12345
connected to server
connected to server
Received message: Hello, Server! from ('127.0.0.1', 48256)
Received from server: Hello from server to ('127.0.0.1', 48256)
Received message: Hello, Server! from ('127.0.0.1', 48256)
Received from server: Hello from server to ('127.0.0.1', 48256)
Received message: Hello, Server! from ('127.0.0.1', 48256)
Received from server: Hello from server to ('127.0.0.1', 48256)
client closed
client closed and joined
server closed
Both processes finished


In [6]:
write_gpio(PMOD_B_PIN_ID, OFF)